In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_lakehouse = "lh_bronze_usaspending"
silver_lakehouse = "lh_silver_usaspending"

table = ["assistance", "contracts"]

# Same composite key as Bronze — Silver mirrors Bronze's grain exactly,
# it's just cleaned, not remodeled.
merge_keys = [
    "award_unique_key",
    "parent_award_id_piid",
    "federal_account_symbol",
    "program_activity_code",
    "program_activity_name",
    "object_class_code",
    "direct_or_reimbursable_funding_source",
    "disaster_emergency_fund_code",
    "submission_period",
    "program_activity_reporting_key",
]

# Text columns worth trimming/cleaning (light touch only, per your requirement)
text_cols_to_clean = [
    "owning_agency_name", "reporting_agency_name", "federal_account_name",
    "program_activity_name", "object_class_name", "awarding_agency_name",
    "awarding_subagency_name", "funding_agency_name", "funding_sub_agency_name",
    "recipient_name", "recipient_name_raw", "recipient_parent_name",
    "recipient_city", "recipient_state", "recipient_country",
]

# String columns that are actually dates and should be cast to DateType
date_cols = [
    "award_base_action_date", "award_latest_action_date",
    "period_of_performance_start_date", "period_of_performance_current_end_date",
    "last_modified_date",
]

StatementMeta(, 1053e4db-3c8d-4a65-bda3-b93551be0dcd, 3, Finished, Available, Finished, False)

In [2]:
def get_watermark(table_name: str):
    row = spark.sql(f"""
        SELECT last_watermark FROM {silver_lakehouse}.dbo.silver_watermark
        WHERE table_name = '{table_name}'
    """).collect()
    return row[0]["last_watermark"] if row else "1900-01-01 00:00:00"


def set_watermark(table_name: str, new_ts):
    spark.sql(f"""
        UPDATE {silver_lakehouse}.dbo.silver_watermark
        SET last_watermark = TIMESTAMP('{new_ts}')
        WHERE table_name = '{table_name}'
    """)

StatementMeta(, 1053e4db-3c8d-4a65-bda3-b93551be0dcd, 4, Finished, Available, Finished, False)

In [3]:
def process_silver(table_short: str):
    bronze_table = f"{bronze_lakehouse}.dbo.bronze_{table_short}"
    silver_table = f"{silver_lakehouse}.dbo.silver_{table_short}"
    watermark_key = f"silver_{table_short}"

    watermark = get_watermark(watermark_key)
    print(f"[{table_short}] Reading Bronze rows newer than {watermark}")

    # Only the rows that changed in Bronze since Silver's last run.
    new_rows = spark.sql(f"""
        SELECT * FROM {bronze_table}
        WHERE _bronze_load_ts > TIMESTAMP('{watermark}')
    """)

    row_count = new_rows.count()
    if row_count == 0:
        print(f"[{table_short}] No new rows since last run — nothing to do.")
        return

    print(f"[{table_short}] {row_count} new/changed rows found. Cleaning...")

    df = new_rows

    # Light cleaning: trim whitespace, normalize blank strings to null
    for c in text_cols_to_clean:
        if c in df.columns:
            df = df.withColumn(c, F.trim(F.col(c)))
            df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

    # Cast date-like string columns to proper DateType
    for c in date_cols:
        if c in df.columns:
            df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

    # Drop exact duplicate rows, guarded: if the key ever stops being unique
    # in future data, stop loudly instead of silently losing rows.
    key_counts = df.groupBy(*merge_keys).count()
    max_collision = key_counts.agg(F.max("count")).collect()[0][0]
    if max_collision and max_collision > 1:
        raise Exception(f"[{table_short}] Merge key collision detected (max {max_collision}) — inspect before proceeding.")
    df = df.dropDuplicates(merge_keys)

    # NEW: timestamp when this row was processed into Silver — this is what
    # Gold uses as its incremental watermark, same role _bronze_load_ts plays for Silver.
    df = df.withColumn("_silver_load_ts", F.current_timestamp())

    if not spark.catalog.tableExists(silver_table):
        print(f"[{table_short}] Creating {silver_table} (first Silver load)")
        df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    else:
        print(f"[{table_short}] Merging into {silver_table}")
        silver = DeltaTable.forName(spark, silver_table)
        join_condition = " AND ".join(f"t.{k} <=> s.{k}" for k in merge_keys)

        (silver.alias("t")
            .merge(df.alias("s"), join_condition)
            .whenMatchedUpdateAll(condition="s.last_modified_date > t.last_modified_date")
            .whenNotMatchedInsertAll()
            .execute())

    # Watermark advances based on Bronze's _bronze_load_ts, not the cleaned data
    new_watermark = new_rows.agg(F.max("_bronze_load_ts")).collect()[0][0]
    set_watermark(watermark_key, new_watermark)
    print(f"[{table_short}] Watermark updated to {new_watermark}")

StatementMeta(, 1053e4db-3c8d-4a65-bda3-b93551be0dcd, 5, Finished, Available, Finished, False)

In [4]:
for t in table:
    try:
        process_silver(t)
    except Exception as e:
        print(f"Failed processing silver_{t}: {e}")

StatementMeta(, 1053e4db-3c8d-4a65-bda3-b93551be0dcd, 6, Finished, Available, Finished, False)

[assistance] Reading Bronze rows newer than 1900-01-01 00:00:00
[assistance] 759666 new/changed rows found. Cleaning...
[assistance] Creating lh_silver_usaspending.dbo.silver_assistance (first Silver load)
[assistance] Watermark updated to 2026-08-31 20:18:57.234412
[contracts] Reading Bronze rows newer than 1900-01-01 00:00:00
[contracts] 188663 new/changed rows found. Cleaning...
[contracts] Creating lh_silver_usaspending.dbo.silver_contracts (first Silver load)
[contracts] Watermark updated to 2026-08-31 20:19:22.429189
